# Stage 5 – Evaluation, threshold tuning and scoring

Compares saved models, tunes the decision threshold, checks calibration and adds a `score_applicants()` helper with low/medium/high risk bands.

Part of **Explainable AI for Credit Risk Prediction**. See the main README for the full pipeline.

In [8]:
# 05_evaluation_and_deployment.ipynb
# =========================================
# Stage 5: Final Evaluation, Threshold Tuning & Deployment Prep
# Project: Explainable AI for Credit Risk Prediction
# =========================================

import os
import json
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

plt.style.use("seaborn-v0_8-whitegrid")

# -----------------------------------------------------
# 0. Paths & config
# -----------------------------------------------------
SAVE_DIR = "../outputs/notebook3_outputs"   # same folder as notebook 3 & 4
os.makedirs(SAVE_DIR, exist_ok=True)

CLEANED_SAMPLE_PATH = os.path.join(SAVE_DIR, "cleaned_sample_for_modelling.csv.gz")
META_PATH           = os.path.join(SAVE_DIR, "feature_metadata.joblib")

LR_PATH      = os.path.join(SAVE_DIR, "logreg.joblib")
RF_PATH      = os.path.join(SAVE_DIR, "random_forest.joblib")
XGB_PATH     = os.path.join(SAVE_DIR, "xgboost.joblib")
HGB_PATH     = os.path.join(SAVE_DIR, "histgb.joblib")
STACK_PATH   = os.path.join(SAVE_DIR, "stacking.joblib")

FINAL_MODEL_PATH      = os.path.join(SAVE_DIR, "final_model_xgb.joblib")
FINAL_CONFIG_JSON     = os.path.join(SAVE_DIR, "final_model_config.json")

TARGET_COL = "default"   # we saved this name in notebook 3

print("Using paths:")
print("  Cleaned sample:", CLEANED_SAMPLE_PATH)
print("  Metadata:      ", META_PATH)
print("  Models:        ", LR_PATH, RF_PATH, XGB_PATH, HGB_PATH, STACK_PATH)
print("  Final model:   ", FINAL_MODEL_PATH)
print("  Config JSON:   ", FINAL_CONFIG_JSON)


# -----------------------------------------------------
# 1. Load data & metadata
# -----------------------------------------------------
if not os.path.exists(CLEANED_SAMPLE_PATH):
    raise FileNotFoundError(
        f"Cleaned sample not found at {CLEANED_SAMPLE_PATH}. Run Notebook 3 first."
    )

df = pd.read_csv(CLEANED_SAMPLE_PATH, compression="infer")
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' missing in cleaned sample.")

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])
print("\n✅ Loaded cleaned sample for modelling.")
print("X shape:", X.shape, "y shape:", y.shape)

# Metadata
if not os.path.exists(META_PATH):
    raise FileNotFoundError(
        f"Metadata file not found at {META_PATH}. Run Notebook 3 first."
    )

meta = joblib.load(META_PATH)
feature_names = meta.get("feature_names", list(X.columns))
numeric_cols  = meta.get("numeric_cols", [])
other_cols    = meta.get("other_cols", [])

# Ensure column order consistent with training
if len(feature_names) == X.shape[1]:
    X = X[feature_names]
else:
    print("\n⚠️ feature_names length mismatch; using X.columns as is.")
    feature_names = list(X.columns)

X_np = X.to_numpy(dtype=float)

print("\nFeature names example:", feature_names[:10])


# -----------------------------------------------------
# 2. Load all trained models
# -----------------------------------------------------
def safe_load(path, name):
    if os.path.exists(path):
        print(f"Loading {name} from {path}")
        return joblib.load(path)
    else:
        print(f"⚠️ {name} file not found at {path}. Skipping.")
        return None

lr      = safe_load(LR_PATH,    "LogisticRegression")
rf      = safe_load(RF_PATH,    "RandomForest")
xgb_clf = safe_load(XGB_PATH,   "XGBoost")
hgb     = safe_load(HGB_PATH,   "HistGradientBoosting")
stack   = safe_load(STACK_PATH, "Stacking ensemble")

models = {
    "LogReg": lr,
    "RandomForest": rf,
    "XGBoost": xgb_clf,
    "HistGB": hgb,
    "Stacking": stack,
}

# -----------------------------------------------------
# 3. Model comparison (ROC AUC, PR AUC, F1 @ 0.5)
# -----------------------------------------------------
print("\n3) Comparing all models on saved sample...")

results = []

for name, model in models.items():
    if model is None:
        continue

    # Probability of default (class 1)
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_np)[:, 1]
    else:
        # fallback for models without predict_proba (not expected here)
        raw = model.decision_function(X_np)
        probs = (raw - raw.min()) / (raw.max() - raw.min())

    y_pred = (probs >= 0.5).astype(int)

    roc  = roc_auc_score(y, probs)
    ap   = average_precision_score(y, probs)
    f1   = f1_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec  = recall_score(y, y_pred)

    print(f"\n=== {name} ===")
    print(classification_report(y, y_pred, digits=4))
    print("Confusion Matrix:\n", confusion_matrix(y, y_pred))
    print(f"ROC AUC: {roc:.6f}, PR AUC: {ap:.6f}, F1: {f1:.6f}, "
          f"Precision: {prec:.6f}, Recall: {rec:.6f}")

    results.append({
        "model": name,
        "roc_auc": roc,
        "pr_auc": ap,
        "f1_at_0.5": f1,
        "precision_at_0.5": prec,
        "recall_at_0.5": rec,
    })

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
print("\nModel comparison summary:")
print(results_df)

results_path = os.path.join(SAVE_DIR, "model_comparison_metrics.csv")
results_df.to_csv(results_path, index=False)
print(f"\nSaved model comparison metrics to: {results_path}")

# Choose champion model – here we pick XGBoost explicitly
if xgb_clf is None:
    raise RuntimeError("XGBoost model not available – cannot proceed as champion.")
print("\nChampion model selected: XGBoost")


# -----------------------------------------------------
# 4. Calibration analysis (reliability curve, Brier score)
# -----------------------------------------------------
print("\n4) Calibration analysis (XGBoost)...")

probs_xgb = xgb_clf.predict_proba(X_np)[:, 1]

brier = brier_score_loss(y, probs_xgb)
print(f"Brier score (lower is better): {brier:.6f}")

# Reliability curve
prob_true, prob_pred = calibration_curve(y, probs_xgb, n_bins=10, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker="o", label="XGBoost")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
plt.xlabel("Predicted default probability")
plt.ylabel("Observed default rate")
plt.title("Calibration plot (reliability curve)")
plt.legend()
calib_path = os.path.join(SAVE_DIR, "xgb_calibration_curve.png")
plt.savefig(calib_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved calibration curve to {calib_path}")

# Histogram of predicted probabilities
plt.figure(figsize=(6, 4))
plt.hist(probs_xgb[y == 0], bins=30, alpha=0.6, label="non-default (0)")
plt.hist(probs_xgb[y == 1], bins=30, alpha=0.6, label="default (1)")
plt.xlabel("Predicted default probability")
plt.ylabel("Count")
plt.title("Distribution of predicted probabilities (XGBoost)")
plt.legend()
dist_path = os.path.join(SAVE_DIR, "xgb_probability_histogram.png")
plt.savefig(dist_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved probability distribution plot to {dist_path}")


# -----------------------------------------------------
# 5. Threshold tuning for business trade-off
# -----------------------------------------------------
print("\n5) Threshold tuning for XGBoost...")

thresholds = np.linspace(0.01, 0.99, 99)
rows = []

for thr in thresholds:
    y_hat = (probs_xgb >= thr).astype(int)
    f1   = f1_score(y, y_hat)
    prec = precision_score(y, y_hat)
    rec  = recall_score(y, y_hat)
    default_rate_flagged = y_hat.mean()

    rows.append({
        "threshold": thr,
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "fraction_flagged_default": default_rate_flagged,
    })

thr_df = pd.DataFrame(rows)

# Best F1 threshold
best_f1_idx = thr_df["f1"].idxmax()
best_f1_thr = thr_df.loc[best_f1_idx, "threshold"]
print(f"\nBest F1 threshold: {best_f1_thr:.3f}")
print(thr_df.loc[best_f1_idx])

# Optional: threshold with high precision (e.g. precision >= 0.7) but some recall
high_prec = thr_df[thr_df["precision"] >= 0.7]
if not high_prec.empty:
    # pick one with highest recall among these
    best_hp_idx = high_prec["recall"].idxmax()
    best_hp_thr = high_prec.loc[best_hp_idx, "threshold"]
    print(f"\nHigh-precision candidate threshold (precision>=0.7): {best_hp_thr:.3f}")
    print(high_prec.loc[best_hp_idx])
else:
    best_hp_thr = best_f1_thr
    print("\nNo threshold with precision>=0.7 found; falling back to best F1 threshold.")

# Save threshold sweep
thr_path = os.path.join(SAVE_DIR, "xgb_threshold_sweep.csv")
thr_df.to_csv(thr_path, index=False)
print(f"\nSaved threshold sweep metrics to {thr_path}")

# Plot F1 vs threshold, Precision vs threshold, Recall vs threshold
plt.figure(figsize=(8, 5))
plt.plot(thr_df["threshold"], thr_df["f1"], label="F1")
plt.plot(thr_df["threshold"], thr_df["precision"], label="Precision")
plt.plot(thr_df["threshold"], thr_df["recall"], label="Recall")
plt.axvline(best_f1_thr, color="red", linestyle="--", label=f"Best F1 thr={best_f1_thr:.2f}")
plt.xlabel("Threshold on predicted default probability")
plt.ylabel("Score")
plt.title("Threshold tuning – XGBoost")
plt.legend()
thr_plot_path = os.path.join(SAVE_DIR, "xgb_threshold_tuning.png")
plt.savefig(thr_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved threshold tuning plot to {thr_plot_path}")

chosen_threshold = float(best_f1_thr)


# -----------------------------------------------------
# 6. Risk bands (low / medium / high) based on predicted probability
# -----------------------------------------------------
print("\n6) Defining risk bands (low / medium / high)...")

# Example bands: you can change these cutoffs for your business logic
low_cut  = 0.10
high_cut = 0.40

risk_band = pd.Series("medium", index=df.index)
risk_band[probs_xgb < low_cut] = "low"
risk_band[probs_xgb >= high_cut] = "high"

df_risk = pd.DataFrame({
    "pred_prob_default": probs_xgb,
    "true_default": y.values,
    "risk_band": risk_band.values,
})

band_stats = (
    df_risk
    .groupby("risk_band")
    .agg(
        count=("true_default", "size"),
        default_rate=("true_default", "mean"),
        avg_pred_prob=("pred_prob_default", "mean"),
    )
)

print("\nRisk band statistics:")
print(band_stats)

band_path = os.path.join(SAVE_DIR, "xgb_risk_band_stats.csv")
band_stats.to_csv(band_path)
print(f"Saved risk band stats to {band_path}")


# -----------------------------------------------------
# 7. Fairness-style slice analysis (simple)
# -----------------------------------------------------
print("\n7) Simple slice analysis (fairness-style)...")

slice_results = []

def evaluate_slice(mask, slice_name):
    if mask.sum() < 100:  # skip tiny slices
        return
    y_slice = y[mask]
    probs_slice = probs_xgb[mask]
    y_pred_slice = (probs_slice >= chosen_threshold).astype(int)

    roc  = roc_auc_score(y_slice, probs_slice)
    ap   = average_precision_score(y_slice, probs_slice)
    f1   = f1_score(y_slice, y_pred_slice)
    prec = precision_score(y_slice, y_pred_slice)
    rec  = recall_score(y_slice, y_pred_slice)
    dr   = y_slice.mean()

    slice_results.append({
        "slice": slice_name,
        "n": int(mask.sum()),
        "base_default_rate": dr,
        "roc_auc": roc,
        "pr_auc": ap,
        "f1": f1,
        "precision": prec,
        "recall": rec,
    })


# Example slices: term (36 vs 60 months) if term_ord exists
if "term_ord" in feature_names:
    term_vals = X["term_ord"].values
    evaluate_slice(term_vals == term_vals.min(), "Term: shorter")
    evaluate_slice(term_vals == term_vals.max(), "Term: longer")

# Grade-based slices if grade_ord exists
if "grade_ord" in feature_names:
    grade_vals = X["grade_ord"].values
    low_grade_mask  = grade_vals <= np.percentile(grade_vals, 33)
    mid_grade_mask  = (grade_vals > np.percentile(grade_vals, 33)) & (grade_vals <= np.percentile(grade_vals, 66))
    high_grade_mask = grade_vals > np.percentile(grade_vals, 66)

    evaluate_slice(low_grade_mask,  "Grade: low (safer)")
    evaluate_slice(mid_grade_mask,  "Grade: medium")
    evaluate_slice(high_grade_mask, "Grade: high (riskier)")

# Income slices if annual_inc present
if "annual_inc" in feature_names:
    inc_vals = X["annual_inc"].values
    low_inc_mask  = inc_vals <= np.percentile(inc_vals, 33)
    mid_inc_mask  = (inc_vals > np.percentile(inc_vals, 33)) & (inc_vals <= np.percentile(inc_vals, 66))
    high_inc_mask = inc_vals > np.percentile(inc_vals, 66)

    evaluate_slice(low_inc_mask,  "Income: low")
    evaluate_slice(mid_inc_mask,  "Income: medium")
    evaluate_slice(high_inc_mask, "Income: high")

slice_df = pd.DataFrame(slice_results)
if not slice_df.empty:
    print("\nSlice performance summary:")
    print(slice_df)
    slice_path = os.path.join(SAVE_DIR, "xgb_slice_analysis.csv")
    slice_df.to_csv(slice_path, index=False)
    print(f"Saved slice analysis to {slice_path}")
else:
    print("No slice metrics created (features missing / sample too small).")


# -----------------------------------------------------
# 8. Stability check via bootstrap (optional)
# -----------------------------------------------------
print("\n8) Stability check via bootstrap (XGBoost)...")

n_boot = 5
rng = np.random.RandomState(42)
boot_metrics = []

for b in range(n_boot):
    idx = rng.choice(np.arange(len(y)), size=len(y), replace=True)
    y_b = y.iloc[idx]
    probs_b = probs_xgb[idx]
    y_hat_b = (probs_b >= chosen_threshold).astype(int)

    roc_b  = roc_auc_score(y_b, probs_b)
    ap_b   = average_precision_score(y_b, probs_b)
    f1_b   = f1_score(y_b, y_hat_b)
    boot_metrics.append({"roc_auc": roc_b, "pr_auc": ap_b, "f1": f1_b})

boot_df = pd.DataFrame(boot_metrics)
print("\nBootstrap metric summary:")
print(boot_df.describe())

boot_path = os.path.join(SAVE_DIR, "xgb_bootstrap_stability.csv")
boot_df.to_csv(boot_path, index=False)
print(f"Saved bootstrap stability metrics to {boot_path}")


# -----------------------------------------------------
# 9. Save final deployment artifacts
# -----------------------------------------------------
print("\n9) Saving final deployment artifacts (model + config)...")

# Save champion model copy as final
joblib.dump(xgb_clf, FINAL_MODEL_PATH)

config = {
    "model_name": "XGBoost",
    "model_path": os.path.abspath(FINAL_MODEL_PATH),
    "chosen_threshold": chosen_threshold,
    "brier_score": float(brier),
    "risk_band_cutoffs": {
        "low_cutoff": low_cut,
        "high_cutoff": high_cut,
    },
    "feature_names": feature_names,
}

with open(FINAL_CONFIG_JSON, "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved final model to: {FINAL_MODEL_PATH}")
print(f"Saved final config to: {FINAL_CONFIG_JSON}")


# -----------------------------------------------------
# 10. Example scoring function (for deployment)
# -----------------------------------------------------
print("\n10) Example scoring function for deployment schema...")

def score_applicants(feature_matrix: pd.DataFrame):
    """
    Score a batch of applicants using the final XGBoost model.

    IMPORTANT:
    - feature_matrix must already have the SAME preprocessing done as Notebook 3
      (same engineered features, same order as `feature_names`).
    - This function is for demonstration in the project write-up.
    """
    # align columns
    missing_cols = [c for c in feature_names if c not in feature_matrix.columns]
    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    X_batch = feature_matrix[feature_names].to_numpy(dtype=float)
    probs = xgb_clf.predict_proba(X_batch)[:, 1]
    preds = (probs >= chosen_threshold).astype(int)

    out = feature_matrix.copy()
    out["pred_default_prob"] = probs
    out["pred_default_label"] = preds
    out["risk_band"] = "medium"
    out.loc[out["pred_default_prob"] < low_cut, "risk_band"] = "low"
    out.loc[out["pred_default_prob"] >= high_cut, "risk_band"] = "high"

    return out

# Small demo with first 5 rows of our sample
demo_scores = score_applicants(X.head(5))
print("\nSample scoring output (first 5 rows):")
print(demo_scores.head())

print("\n✅ Notebook 5 completed: final evaluation, threshold tuning, and deployment artifacts are ready.")
print("Check outputs in:", SAVE_DIR)


Using paths:
  Cleaned sample: ./notebook3_outputs\cleaned_sample_for_modelling.csv.gz
  Metadata:       ./notebook3_outputs\feature_metadata.joblib
  Models:         ./notebook3_outputs\logreg.joblib ./notebook3_outputs\random_forest.joblib ./notebook3_outputs\xgboost.joblib ./notebook3_outputs\histgb.joblib ./notebook3_outputs\stacking.joblib
  Final model:    ./notebook3_outputs\final_model_xgb.joblib
  Config JSON:    ./notebook3_outputs\final_model_config.json

✅ Loaded cleaned sample for modelling.
X shape: (40000, 85) y shape: (40000,)

Feature names example: ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high']
Loading LogisticRegression from ./notebook3_outputs\logreg.joblib
Loading RandomForest from ./notebook3_outputs\random_forest.joblib
Loading XGBoost from ./notebook3_outputs\xgboost.joblib
Loading HistGradientBoosting from ./notebook3_outputs\histgb.joblib
Loading Stacking ensem